# 10. Information Extraction from an Invoice Image

Send a supplied invoice image to a vision-capable OpenAI model and return validated structured data.

## Workflow

Invoice image -> Base64 data URL -> multimodal message -> structured extraction -> arithmetic validation -> Pandas record.

### 1. Set up the API key and model

This cell imports the required classes, reads the API key securely, and selects the model without exposing credentials.

**Expected result:** No model output is produced; the environment becomes ready for later API calls. Read the output before continuing to the next cell.

In [ ]:
import os, getpass
from langchain_openai import ChatOpenAI
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ")
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

### 2. Prepare the next processing step

This cell prepares the variables, functions, or validation logic required by the next stage of the example.

**Expected result:** The cell defines reusable objects or prints a small verification result. Read the output before continuing to the next cell.

In [ ]:
from pathlib import Path
from IPython.display import Image,display
image_path=Path("assets/invoice_sample.png")
if not image_path.exists(): image_path=Path("../assets/invoice_sample.png")
display(Image(filename=str(image_path),width=600))

### 3. Prepare and submit the invoice image

This cell prepares the invoice image as multimodal input and sends both instructions and image content to the model.

**Expected result:** The vision-capable model returns the requested invoice fields. Read the output before continuing to the next cell.

In [ ]:
import base64
mime="image/png"
encoded=base64.b64encode(image_path.read_bytes()).decode("utf-8")
data_url=f"data:{mime};base64,{encoded}"
print("Image prepared; Base64 characters:",len(encoded))

### 4. Define and validate structured output

This cell defines a Pydantic schema and configures structured output so model results have predictable fields and types.

**Expected result:** A validated Python object or dictionary is returned instead of unstructured text. Read the output before continuing to the next cell.

In [ ]:
from pydantic import BaseModel
class Invoice(BaseModel):
    invoice_number:str; vendor:str; invoice_date:str; due_date:str
    currency:str; subtotal:float; tax:float; total:float; payment_terms:str

vision=ChatOpenAI(model=MODEL_NAME,temperature=0).with_structured_output(Invoice)

### 5. Prepare and submit the invoice image

This cell prepares the invoice image as multimodal input and sends both instructions and image content to the model.

**Expected result:** The vision-capable model returns the requested invoice fields. Read the output before continuing to the next cell.

In [ ]:
message={"role":"user","content":[
    {"type":"text","text":"Extract all invoice fields. Use numbers without commas or currency symbols. Do not invent missing values."},
    {"type":"image_url","image_url":{"url":data_url}}
]}
invoice=vision.invoke([message])
print(invoice.model_dump_json(indent=2))

### 6. Validate the generated result

This cell checks important business and safety rules before the generated result is accepted.

**Expected result:** Boolean checks or assertions confirm whether the result is valid. Read the output before continuing to the next cell.

In [ ]:
arithmetic_ok=round(invoice.subtotal+invoice.tax,2)==round(invoice.total,2)
print("Subtotal + tax = total:",arithmetic_ok)
assert invoice.invoice_number and invoice.vendor
assert invoice.subtotal>=0 and invoice.tax>=0 and invoice.total>=0

### 7. Organize the results with Pandas

This cell organizes the extracted or generated values into a Pandas table for inspection and analysis.

**Expected result:** A structured DataFrame or summary table is displayed. Read the output before continuing to the next cell.

In [ ]:
import pandas as pd
invoice_df=pd.DataFrame([invoice.model_dump()])
display(invoice_df)

## Production safeguards

Validate currency, dates, totals and supplier identity. Detect duplicate invoice numbers, protect financial data, retain the source image and require approval before payment.

## Exercise

Change one visible amount in a copy of the image, extract again, and test whether arithmetic validation catches the inconsistency. Try a rotated or lower-quality image.